pip install command

In [ ]:
# pip install ollama chromadb pypdf pdfplumber python-docx Pillow gradio --break-system-packages

Imports

In [ ]:
from ollama import Client
import chromadb
import pdfplumber
from docx import Document
from pathlib import Path
import base64
import json
import threading
import webbrowser
from http.server import HTTPServer, BaseHTTPRequestHandler

check for files

In [ ]:
docs_folder = Path("tobroco_manuals")
pdf_files = list(docs_folder.glob("*.pdf"))
docx_files = list(docs_folder.glob("*.docx"))
all_files = pdf_files + docx_files

Connection setup

In [ ]:
ollama_client = Client(host="http://127.0.0.1:11434")
chroma_client = chromadb.PersistentClient(path="chroma_db")

LLM_MODEL =  "llama3.3:70b-instruct-q5_K_M" # "llama3.1:8b-instruct-q8_0"  
REBUILD_DATABASE = False

if REBUILD_DATABASE:
    try:
        chroma_client.delete_collection(name="tobroco_manuals")
    except:
        pass
    collection = chroma_client.create_collection(name="tobroco_manuals")
else:
    try:
        collection = chroma_client.get_collection(name="tobroco_manuals")
    except:
        collection = chroma_client.create_collection(name="tobroco_manuals")

Process function definition

In [ ]:
def extract_pdf_text(pdf_path):
    documents = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text and text.strip():
                documents.append({
                    "content": text,
                    "metadata": {
                        "source": Path(pdf_path).name,
                        "page": page_num,
                        "type": "pdf"
                    }
                })
    return documents

def extract_docx_text(docx_path):
    doc = Document(docx_path)
    documents = []
    current_section = []
    section_num = 1
    word_count = 0
    
    for para in doc.paragraphs:
        if para.text.strip():
            current_section.append(para.text)
            word_count += len(para.text.split())
            
            if word_count >= 500:
                documents.append({
                    "content": "\n".join(current_section),
                    "metadata": {
                        "source": Path(docx_path).name,
                        "page": section_num,
                        "type": "docx"
                    }
                })
                current_section = []
                word_count = 0
                section_num += 1
    
    if current_section:
        documents.append({
            "content": "\n".join(current_section),
            "metadata": {
                "source": Path(docx_path).name,
                "page": section_num,
                "type": "docx"
            }
        })
    return documents

# Unused, commented-out image extraction prototype omitted.

def extract_text_from_document(file_path):
    file_path = Path(file_path)
    if file_path.suffix.lower() == '.pdf':
        return extract_pdf_text(file_path)
    elif file_path.suffix.lower() in ['.docx', '.doc']:
        return extract_docx_text(file_path)  # image analysis disabled
    else:
        return []

def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

def create_embedding(text):
    response = ollama_client.embeddings(
        model="mxbai-embed-large:latest",
        prompt=text
    )
    return response["embedding"]

def process_documents(file_paths):
    doc_id = 0
    for file_path in file_paths:
        documents = extract_text_from_document(file_path)
        for doc in documents:
            chunks = chunk_text(doc["content"])
            for chunk_idx, chunk in enumerate(chunks):
                try:
                    embedding = create_embedding(chunk)
                    collection.add(
                        embeddings=[embedding],
                        documents=[chunk],
                        metadatas=[{
                            "source": doc["metadata"]["source"],
                            "page": doc["metadata"]["page"],
                            "chunk": chunk_idx,
                            "type": doc["metadata"]["type"]
                        }],
                        ids=[f"doc_{doc_id}"]
                    )
                    doc_id += 1
                except:
                    continue
                
def search_manuals(query, top_k=5):
    query_embedding = create_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    citations = []
    for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        citations.append({
            "rank": i + 1,
            "text": doc,
            "source": metadata['source'],
            "page": metadata['page']
        })
    return citations

LLM answer

In [ ]:
def answer_query(query):
    relevant_chunks = search_manuals(query, top_k=3)
    context = "\n\n".join([
        f"[Source: {chunk['source']}, Page {chunk['page']}]\n{chunk['text']}"
        for chunk in relevant_chunks
    ])
    
    prompt = f"""You are an engineering manual assistant. Based on the following manual excerpts, answer the question.

CRITICAL RULES:
1. ALWAYS cite the exact source and page number for each piece of information
2. If the text contains a section number or header (like "5.3 Material Standards"), INCLUDE IT in the citation
3. Return the EXACT text from the manual, do NOT summarize or paraphrase
4. When citing multiple sections from the SAME page of the SAME document, combine them under ONE page reference
5. Format citations as:
   "According to [Document Name]:
   
   Page [Y]: [quote 1]
   [quote 2]
   [quote 3]
   
   Page [Z]: [quote]"
6. For different documents, cite each separately with blank lines between
7. If no section header is visible, use: "Page [Number]: [exact quote]"
8. Group all quotes from the same page together to avoid repeating "Page X" multiple times

Manual Excerpts:
{context}

Question: {query}

Provide the exact relevant text from the manuals with proper citations:"""
    
    stream = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        options={"num_gpu": 99, "num_thread": 8}
    )
    
    return {
        "answer": stream,
        "sources": relevant_chunks
    }

main

In [ ]:
docs_folder = Path("temp_pdfs")
all_files = list(docs_folder.glob("*.pdf"))

if REBUILD_DATABASE and all_files:
    process_documents(all_files)

UI

In [ ]:
class RequestHandler(BaseHTTPRequestHandler):

    def do_GET(self):
        if self.path == '/':
            file_path = Path("tobroco_assistant.html")
            content_type = 'text/html'
        elif self.path == '/logo.png':
            file_path = Path("logo.png")
            content_type = 'image/png'
        else:
            self.send_error(404)
            return

        if file_path.exists():
            content = file_path.read_bytes()
            self.send_response(200)
            self.send_header('Content-type', content_type)
            self.send_header('Content-Length', len(content))
            self.end_headers()
            self.wfile.write(content)
        else:
            self.send_error(404, f"{file_path.name} not found")

    def do_POST(self):
        if self.path == '/api/stream':
            content_length = int(self.headers['Content-Length'])
            body = json.loads(self.rfile.read(content_length).decode())
            question = body.get('question', '').strip()

            self.send_response(200)
            self.send_header('Content-type', 'text/event-stream')
            self.send_header('Cache-Control', 'no-cache')
            self.send_header('X-Accel-Buffering', 'no')
            self.end_headers()

            def send_event(data):
                line = f"data: {json.dumps(data)}\n\n"
                self.wfile.write(line.encode())
                self.wfile.flush()

            if not question:
                send_event({"type": "token", "text": "Please enter a question."})
                send_event({"type": "done"})
                return

            try:
                result = answer_query(question)

                send_event({"type": "sources", "data": [
                    {"source": c["source"], "page": c["page"]} for c in result["sources"]
                ]})

                for chunk in result["answer"]:
                    token = chunk["message"]["content"]
                    if token:
                        send_event({"type": "token", "text": token})

                send_event({"type": "done"})

            except Exception as e:
                send_event({"type": "token", "text": f"Error: {str(e)}"})
                send_event({"type": "done"})

    def log_message(self, format, *args):
        pass


PORT = 5000
server = HTTPServer(('127.0.0.1', PORT), RequestHandler)

print(f"\n{'='*70}")
print(f"🚀 Tobroco-Giant Engineering Assistant")
print(f"{'='*70}")
print(f"\n🌐 Open your browser: http://127.0.0.1:{PORT}")
print(f"\n⏹️  Press Ctrl+C to stop the server\n")

threading.Timer(1.0, lambda: webbrowser.open(f'http://127.0.0.1:{PORT}')).start()

try:
    server.serve_forever()
except KeyboardInterrupt:
    print("\n✓ Server stopped")
    server.server_close()